In [1]:
import sys, pathlib as pl; sys.path.insert(0, str(pl.Path.cwd().parents[1])); from fig_utils import fig_csv
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
mpl.use("Cairo")  # for saving SVGs that Affinity Designer can parse

import candas as can
import gumbi as gmb
from candas.test import FluorescenceData, QuantStudio

import pathlib as pl
code_pth = pl.Path.cwd()  # for running in Jupyter
# code_pth = pl.Path(__file__)  # for running in terminal
fig_pth = code_pth.parent
data_pth = fig_pth / 'data'
graph_pth = fig_pth / 'graphics'
graph_pth.mkdir(exist_ok=True)

gen_pth = fig_pth / 'generated'
gen_pth.mkdir(exist_ok=True)

plt.style.use(str(can.style.breve))

%config InlineBackend.figure_format = 'retina'

WARNING (theano.link.c.cmodule): install mkl with `conda install mkl-service`: No module named 'mkl'
/opt/anaconda3/envs/can_manuscript/lib/python3.9/site-packages/arviz/data/base.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
from utils import savefig

In [3]:
width = 5.2
height = 0.7
figsize = (width, height)
ticklabelsize = 6
labelsize = 6
titlesize = labelsize

# Set rcParams for plotting
mpl.rc("xtick", labelsize=ticklabelsize)
mpl.rc("ytick", labelsize=ticklabelsize)
mpl.rc("axes", labelsize=labelsize, titlesize=titlesize, linewidth=0.5)

mar_l = 0.11
mar_r = 0.15
mar_t = 0.5
mar_b = 2.5


def format_fig(
    fig, figsize=figsize, mar_l=mar_l, mar_r=mar_r, mar_t=mar_t, mar_b=mar_b, **kwargs
):
    """Adjust margins of subplots using figsize"""
    height, width = figsize
    fig.set_size_inches(figsize)

    plt.subplots_adjust(
        left=mar_l / width,
        right=1 - mar_r / width,
        top=1 - mar_t / height,
        bottom=mar_b / height,
        **kwargs
    )

    for ax in fig.get_axes():
        ax.tick_params(which="both", length=1.0, width=0.5)

# JG075A: EGFR Blocker Stoichiometry

In [4]:
cmax = 40

JG075A = (
    QuantStudio(data_pth / "JG075A EGFR Blocker Stoichiometry.xlsx", "JG075A")
    .import_data()
    .format_reactions()
    .index_reactions()
    .subtract_background()
    .normalize_reactions(cmax=cmax)
    .trim_reactions()
    #     .invert_fluorophore('HEX')
)
JG075A.reactions.wide = JG075A.reactions.wide.drop(columns=["Stage", "Derivative"])
JG075A.reactions.signal_columns = [
    col for col in JG075A.reactions.signal_columns if col != "Derivative"
]

# # Denote reaction conditions
JG075A.reactions.wide = (
    JG075A.reactions.wide.merge(
        pd.read_csv(data_pth / "JG075A Plate Map.csv")
    )  # [['Sample','WT Conc','Competitor Conc','Primer Conc']])
    .assign(lg10_Blocker=lambda df: np.log10(df["Blocker μM"]))
    .replace({"lg10_Blocker": {-np.inf: -2}})
    #     .drop(columns=['CT'])
    .query("Well != 73")  # Outlier, weird drift in baseline
)

JG075A.reactions.neaten()

/opt/anaconda3/envs/can_manuscript/lib/python3.9/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


,Experiment,Well,Target,lg10_Copies,Outlier,WellPosition,Copies,Reporter,Sample,Task,Comments,CT,Reaction,Blocker,Blocker μM,WellName,lg10_Blocker,Cycle,Fluorescence,Corr_Fluorescence
0,JG075A,1,S075_WT,8.0,False,A1,100000000.0,EVAGREEN,JG075A_001,STANDARD,,56.821556,0,MMMMx,3.162278,A1,0.5,3,-0.006387,0.089428
1,JG075A,1,S075_WT,8.0,False,A1,100000000.0,EVAGREEN,JG075A_001,STANDARD,,56.821556,0,MMMMx,3.162278,A1,0.5,4,-0.005734,0.045397
2,JG075A,1,S075_WT,8.0,False,A1,100000000.0,EVAGREEN,JG075A_001,STANDARD,,56.821556,0,MMMMx,3.162278,A1,0.5,5,-0.004740,0.002104
3,JG075A,1,S075_WT,8.0,False,A1,100000000.0,EVAGREEN,JG075A_001,STANDARD,,56.821556,0,MMMMx,3.162278,A1,0.5,6,-0.001089,-0.035434
4,JG075A,1,S075_WT,8.0,False,A1,100000000.0,EVAGREEN,JG075A_001,STANDARD,,56.821556,0,MMMMx,3.162278,A1,0.5,7,0.000279,-0.077918
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12235,JG075A,375,S075_SNV,1.0,False,P15,10.0,EVAGREEN,JG075A_375,STANDARD,,20.933987,238,None,0.000000,P15,-2.0,46,0.945859,1.537327
12236,JG075A,375,S075_SNV,1.0,False,P15,10.0,EVAGREEN,JG075A_375,STANDARD,,20.933987,238,None,0.000000,P15,-2.0,47,0.945898,1.524482
12237,JG075A,375,S075_SNV,1.0,False,P15,10.0,EVAGREEN,JG075A_375,STANDARD,,20.933987,238,None,0.000000,P15,-2.0,48,0.949281,1.518882
12238,JG075A,375,S075_SNV,1.0,False,P15,10.0,EVAGREEN,JG075A_375,STANDARD,,20.933987,238,None,0.000000,P15,-2.0,49,0.952237,1.512355


In [5]:
target_palette = list(zip(["S075_WT", "S075_SNV"], ["Purples", "Greens"]))
blockers = ["L-MMMMx", "MMMMx"]

In [6]:
hue = "lg10_Blocker"
extent = np.max(np.abs(JG075A.reactions.wide[hue]))
norm = mpl.colors.Normalize(
    vmin=JG075A.reactions.wide[hue].min(), vmax=JG075A.reactions.wide[hue].max()
)


for target, palette in target_palette:
    data = JG075A.reactions.data
    data = data[
        (data.lg10_Copies >= 4)
        & (data.Blocker == "L-MMMMx")
        & (data.Target == target)
        & (data.Well != 73)
    ]

    g = sns.relplot(
        data=data,
        x="Cycle",
        y="Fluorescence",
        col="lg10_Copies",
        # row="Blocker",
        units="Reaction",
        hue=hue,
        legend=False,
        palette=palette,
        hue_norm=norm,
        kind="line",
        estimator=None,
        height=3,
        aspect=1.5,
        linewidth=1,
        col_order=[8.0, 7.0, 6.0, 5.0, 4.0],
        facet_kws={"margin_titles": True, "despine": False},
    )
    if target ==  "S075_WT":
        data.to_csv(fig_csv('sweet_carp'))
    else:
        data.to_csv(fig_csv('quiet_urchin'))
    

    g.refline(y=0, color="k", linestyle="-", linewidth= 0.5)
    g.refline(y=0.1, color="k", linestyle=":", linewidth= 0.5)
    g.set(ylim=[-0.1, 1.1], xlim=[0, 60], yticks=[0, 0.5, 1])
    plt.tight_layout()
    
    fig = plt.gcf()

    if target == "S075_WT":

        g.set_titles(col_template="")
        g.set(xticks=[0, 20, 40, 60])
        g.set_xlabels('Cycle', fontsize=labelsize, labelpad=0.5)
        format_fig(
            fig,
            figsize=(width, height),
            mar_l=0.1,
            mar_r=0.02,
            mar_t=0.25,
            mar_b=1.95,
            wspace=0.2
        )
    else:
        g.set_titles(col_template="{col_name:.1f}", fontsize=labelsize, pad=labelsize*0.5)
        g.set(xticks=[])
        g.set_xlabels('')
        format_fig(
            fig,
            figsize=(width, height),
            mar_l=0.1,
            mar_r=0.02,
            mar_t=1.5,
            mar_b=0.65,
            wspace=0.2
        )
        
    
    # multicolor_ylabel(
    #     fig,
    #     (target, "\nFluorescence"),
    #     (palette.lower().removesuffix('s'), "k"),
    #     axis="y",
    #     ybbox=(0, 0.2),
    #     fontsize=labelsize
    # )
    
    ax = fig.axes[0]
    ax.text(
        x=-0.45,
        y=0.5,
        s=target.removeprefix("S075_"),
        fontsize=labelsize,
        color={"S075_WT": "purple", "S075_SNV": "green"}[target],
        transform=ax.transAxes,
        ha="right",
        va="center",
        rotation=90,
    )
    

    alias = {"S075_WT": "quiet_urchin", "S075_SNV": "sweet_carp"}[target]

    #savefig(fig, alias=alias)